In [1]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model(
    r"C:\Users\j9698\OneDrive\Desktop\plant_disease_recog_model_pwp.keras"
)

# Check model output shape
print("Output shape:", model.output_shape)
print("Number of classes:", model.output_shape[-1])

# Check if class names are saved inside the model
print("\nModel config summary:")
print(model.summary())

Output shape: (None, 39)
Number of classes: 39

Model config summary:


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb4 (Functional)     │ (None, 5, 5, 1792)     │    17,673,823 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1792)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 39)             │        69,927 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,807,318 (201.44 MB)

 Trainable params: 17,531,783 (66.88 MB)

 Non-trainable params: 211,967 (828.00 KB)

 Optimizer params: 35,063,568 (133.76 MB)

None


In [3]:
# Check if you have any class names saved from training
import os
for root, dirs, files in os.walk(r"C:\Users\j9698\OneDrive\Desktop"):
    for f in files:
        if "class" in f.lower() and f.endswith(".json"):
            print(os.path.join(root, f))

In [4]:
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model(
    r"C:\Users\j9698\OneDrive\Desktop\plant_disease_recog_model_pwp.keras"
)

# Test with your tomato leaf image
img_path = r"C:\Users\j9698\OneDrive\Desktop\agrosense\uploads\fa72e8ed-3eb2-4d3d-a1fd-f099b43222ac_leaf.jpg"

img  = Image.open(img_path).convert("RGB").resize((160, 160))
arr  = np.array(img, dtype=np.float32)
arr  = tf.keras.applications.efficientnet.preprocess_input(arr)
arr  = np.expand_dims(arr, axis=0)

preds = model.predict(arr, verbose=0)[0]

print("Top 5 predictions:")
top5 = np.argsort(preds)[::-1][:5]
for i in top5:
    print(f"  Index {i:2d}: {preds[i]*100:.1f}%")

print(f"\nTotal classes: {len(preds)}")

Top 5 predictions:
  Index 31: 100.0%
  Index 33: 100.0%
  Index 30: 100.0%
  Index 21: 100.0%
  Index  4: 100.0%

Total classes: 39


In [5]:
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model(
    r"C:\Users\j9698\OneDrive\Desktop\plant_disease_recog_model_pwp.keras"
)

img_path = r"C:\Users\j9698\OneDrive\Desktop\agrosense\uploads\fa72e8ed-3eb2-4d3d-a1fd-f099b43222ac_leaf.jpg"

img = Image.open(img_path).convert("RGB").resize((160, 160))
arr = np.array(img, dtype=np.float32) / 255.0  # Simple normalization
arr = np.expand_dims(arr, axis=0)

preds = model.predict(arr, verbose=0)[0]

print("Top 5 predictions:")
top5 = np.argsort(preds)[::-1][:5]
for i in top5:
    print(f"  Index {i:2d}: {preds[i]*100:.2f}%")

Top 5 predictions:
  Index 11: 100.00%
  Index  9: 100.00%
  Index  7: 100.00%
  Index 16: 100.00%
  Index 19: 100.00%


In [6]:
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model(
    r"C:\Users\j9698\OneDrive\Desktop\plant_disease_recog_model_pwp.keras"
)

img_path = r"C:\Users\j9698\OneDrive\Desktop\agrosense\uploads\fa72e8ed-3eb2-4d3d-a1fd-f099b43222ac_leaf.jpg"

# Try all 3 preprocessing methods and print raw logits
img = Image.open(img_path).convert("RGB").resize((160, 160))
arr = np.array(img, dtype=np.float32)

print("=== Method 1: /255 normalization ===")
a1 = np.expand_dims(arr / 255.0, axis=0)
p1 = model.predict(a1, verbose=0)[0]
print("Min:", round(float(p1.min()), 4), "Max:", round(float(p1.max()), 4))
print("Sum:", round(float(p1.sum()), 4))
top3 = np.argsort(p1)[::-1][:3]
for i in top3:
    print(f"  Index {i}: {p1[i]*100:.4f}%")

print("\n=== Method 2: EfficientNet preprocess ===")
a2 = np.expand_dims(tf.keras.applications.efficientnet.preprocess_input(arr.copy()), axis=0)
p2 = model.predict(a2, verbose=0)[0]
print("Min:", round(float(p2.min()), 4), "Max:", round(float(p2.max()), 4))
print("Sum:", round(float(p2.sum()), 4))
top3 = np.argsort(p2)[::-1][:3]
for i in top3:
    print(f"  Index {i}: {p2[i]*100:.4f}%")

print("\n=== Method 3: Raw pixels 0-255 ===")
a3 = np.expand_dims(arr, axis=0)
p3 = model.predict(a3, verbose=0)[0]
print("Min:", round(float(p3.min()), 4), "Max:", round(float(p3.max()), 4))
print("Sum:", round(float(p3.sum()), 4))
top3 = np.argsort(p3)[::-1][:3]
for i in top3:
    print(f"  Index {i}: {p3[i]*100:.4f}%")

print("\n=== Method 4: /127.5 - 1 ===")
a4 = np.expand_dims((arr / 127.5) - 1.0, axis=0)
p4 = model.predict(a4, verbose=0)[0]
print("Min:", round(float(p4.min()), 4), "Max:", round(float(p4.max()), 4))
print("Sum:", round(float(p4.sum()), 4))
top3 = np.argsort(p4)[::-1][:3]
for i in top3:
    print(f"  Index {i}: {p4[i]*100:.4f}%")

=== Method 1: /255 normalization ===
Min: 0.0 Max: 1.0
Sum: 20.7321
  Index 11: 100.0000%
  Index 9: 100.0000%
  Index 7: 99.9997%

=== Method 2: EfficientNet preprocess ===
Min: 0.0001 Max: 1.0
Sum: 17.9688
  Index 31: 100.0000%
  Index 33: 100.0000%
  Index 30: 99.9974%

=== Method 3: Raw pixels 0-255 ===
Min: 0.0001 Max: 1.0
Sum: 17.9688
  Index 31: 100.0000%
  Index 33: 100.0000%
  Index 30: 99.9974%

=== Method 4: /127.5 - 1 ===
Min: 0.0001 Max: 1.0
Sum: 20.962
  Index 11: 100.0000%
  Index 9: 100.0000%
  Index 16: 99.9997%


In [7]:
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model(
    r"C:\Users\j9698\OneDrive\Desktop\plant_disease_recog_model_pwp.keras"
)

# Build a model that outputs raw logits (before softmax)
logit_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.layers[-2].output  # Second to last layer
)

img_path = r"C:\Users\j9698\OneDrive\Desktop\agrosense\uploads\fa72e8ed-3eb2-4d3d-a1fd-f099b43222ac_leaf.jpg"
img = Image.open(img_path).convert("RGB").resize((160, 160))
arr = np.array(img, dtype=np.float32) / 255.0
arr = np.expand_dims(arr, axis=0)

logits = logit_model.predict(arr, verbose=0)[0]
print("Raw logits (top 10):")
top10 = np.argsort(logits)[::-1][:10]
for i in top10:
    print(f"  Index {i:2d}: {logits[i]:.4f}")

print(f"\nLogit min: {logits.min():.4f}")
print(f"Logit max: {logits.max():.4f}")

# Apply softmax manually
exp_logits = np.exp(logits - logits.max())
softmax = exp_logits / exp_logits.sum()
print(f"\nManual softmax sum: {softmax.sum():.4f}")
print("\nTop 5 after manual softmax:")
top5 = np.argsort(softmax)[::-1][:5]
for i in top5:
    print(f"  Index {i:2d}: {softmax[i]*100:.2f}%")

Raw logits (top 10):
  Index 1612: 2.1836
  Index 94: 2.0312
  Index 1359: 1.8623
  Index 1028: 1.6288
  Index 546: 1.5731
  Index 1020: 1.5634
  Index 20: 1.5561
  Index 276: 1.4524
  Index 783: 1.4413
  Index 1307: 1.2518

Logit min: -0.2625
Logit max: 2.1836

Manual softmax sum: 1.0000

Top 5 after manual softmax:
  Index 1612: 0.49%
  Index 94: 0.42%
  Index 1359: 0.36%
  Index 1028: 0.28%
  Index 546: 0.27%


In [10]:
# Fix for the error
for i, layer in enumerate(model.layers):
    try:
        shape = str(layer.output.shape)
    except:
        shape = "N/A"
    print(f"{i:2d} {layer.name:40s} {shape}")

 0 input_layer_6                            (None, 160, 160, 3)
 1 efficientnetb4                           N/A
 2 global_average_pooling2d_1               (None, 1792)
 3 dropout_3                                (None, 1792)
 4 dense_1                                  (None, 39)


In [4]:
import tensorflow as tf
import numpy as np
from PIL import Image

# Load model fresh
model = tf.keras.models.load_model(
    r"C:\Users\j9698\OneDrive\Desktop\plant_disease_recog_model_pwp.keras",
    compile=False
)

img_path = r"C:\Users\j9698\OneDrive\Desktop\agrosense\uploads\fa72e8ed-3eb2-4d3d-a1fd-f099b43222ac_leaf.jpg"
img = Image.open(img_path).convert("RGB").resize((160, 160))
arr = np.array(img, dtype=np.float32) / 255.0
arr = np.expand_dims(arr, axis=0)

# Get raw predictions
preds = model(arr, training=False).numpy()[0]
print("Raw output shape:", preds.shape)
print("Sum:", round(float(preds.sum()), 4))
print("Min:", round(float(preds.min()), 4))
print("Max:", round(float(preds.max()), 4))

# Apply temperature scaling to soften predictions
temperature = 5.0
scaled = np.log(preds + 1e-10) / temperature
exp_scaled = np.exp(scaled - scaled.max())
softmax_scaled = exp_scaled / exp_scaled.sum()

print("\nAfter temperature scaling:")
print("Sum:", round(float(softmax_scaled.sum()), 4))
top5 = np.argsort(softmax_scaled)[::-1][:5]
for i in top5:
    print(f"  Index {i:2d}: {softmax_scaled[i]*100:.2f}%")

AttributeError: module 'keras' has no attribute 'KerasTensor'

In [1]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"]  = "3"

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, callbacks
from PIL import Image

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32

# ── Find the data folder ───────────────────────────────
DATA_DIR = None
for root, dirs, files in os.walk("data/plantvillage"):
    for d in dirs:
        if "Apple___Apple_scab" in os.listdir(os.path.join(root, d)) or \
           any("Apple___Apple_scab" in sd for sd in os.listdir(os.path.join(root, d)) if os.path.isdir(os.path.join(root, d, sd))):
            DATA_DIR = os.path.join(root, d)
            break
    if DATA_DIR:
        break

# If not found automatically use this
if not DATA_DIR:
    DATA_DIR = "data/plantvillage"

print("Using data dir:", DATA_DIR)
print("Classes found:", len(os.listdir(DATA_DIR)))

# ── Load dataset ───────────────────────────────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)
print("First 5:", CLASS_NAMES[:5])

# Save class names
import json
os.makedirs("models/disease_model", exist_ok=True)
with open("models/disease_model/class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)
print("Class names saved!")

# ── Preprocessing ──────────────────────────────────────
def preprocess(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().shuffle(2000).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

# ── Build model ────────────────────────────────────────
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
], name="augmentation")

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
x       = augment(inputs)
base    = tf.keras.applications.EfficientNetB4(
              include_top=False,
              weights="imagenet",
              input_shape=(*IMG_SIZE, 3))
base.trainable = False
x       = base(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation="relu")(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model   = tf.keras.Model(inputs, outputs)

print("Model built! Parameters:", model.count_params())

# ── Phase A: Train head ────────────────────────────────
print("\n=== Phase A: Training head ===")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
history_a = model.fit(
    train_ds, epochs=10,
    validation_data=val_ds,
    callbacks=[
        callbacks.EarlyStopping(
            patience=3, restore_best_weights=True, monitor="val_accuracy"),
        callbacks.ReduceLROnPlateau(
            patience=2, factor=0.5, monitor="val_loss"),
    ]
)
print(f"Phase A best: {max(history_a.history['val_accuracy'])*100:.1f}%")

# ── Phase B: Fine-tune ─────────────────────────────────
print("\n=== Phase B: Fine-tuning ===")
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
history_b = model.fit(
    train_ds, epochs=20,
    validation_data=val_ds,
    callbacks=[
        callbacks.EarlyStopping(
            patience=5, restore_best_weights=True, monitor="val_accuracy"),
        callbacks.ReduceLROnPlateau(
            patience=3, factor=0.3, monitor="val_loss"),
        callbacks.ModelCheckpoint(
            "models/disease_model/disease_classifier.keras",
            save_best_only=True, monitor="val_accuracy"
        ),
    ]
)

best_acc = max(history_b.history["val_accuracy"])
print(f"\nBest val accuracy: {best_acc*100:.1f}%")

# ── Save everything ────────────────────────────────────
model.save("models/disease_model/disease_classifier.keras")
print("Model saved!")
print("Class names saved to models/disease_model/class_names.json")

Using data dir: data/plantvillage\plantvillage dataset\color
Classes found: 38
Found 54305 files belonging to 38 classes.
Using 43444 files for training.
Found 54305 files belonging to 38 classes.
Using 10861 files for validation.
Number of classes: 38
First 5: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']
Class names saved!
71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Model built! Parameters: 18149765

=== Phase A: Training head ===
Epoch 1/10
 101/1358 ━━━━━━━━━━━━━━━━━━━━ 31:59 2s/step - accuracy: 0.0533 - loss: 4.7199

KeyboardInterrupt: 